# ACC 380K — A random forest that writes stories

A language model does one small thing, then does it again. It looks at the words so far, predicts the next word, and appends that word. ChatGPT is that loop at a huge scale. This notebook builds the same loop with `RandomForestClassifier`, the same estimator as the grades and digits examples. Here the label is the next word.

**Notebook version 0.7.0 · November 12, 2026**

Run the cells from top to bottom. In Colab, choose **File → Save a copy in Drive** if you want your own copy. There is no API key and no file to upload. One cell downloads a public file of children's stories. A full run takes about four minutes on a free runtime.

## What you should be able to do

- Turn a story into rows: the last few words in, the next word as the label.
- See the one-hot table the forest actually trains on.
- Read `predict_proba` as a ranked list of next words.
- Generate text by predicting, appending, shifting, and repeating, then add randomness with a temperature.
- Say what the forest can and cannot see, and why that limits the story.

## Setup

`K` is how many previous tokens the forest sees. A token is a lowercase word or a punctuation mark. `<BOS>` means "beginning of the story." It fills the empty positions before the story starts. `<UNK>` stands for a rare word outside the forest's vocabulary.

`random_state` is fixed, so a second run gives the same forest and the same story.

In [ ]:
import hashlib
import re
import time
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder

K = 4
BOS = "<BOS>"
UNK = "<UNK>"
CONTEXT_COLUMNS = ["t_minus_" + str(step) for step in range(K, 0, -1)]
RANDOM_STATE = 38026

print("Context columns:", CONTEXT_COLUMNS)

## Download simple stories

The stories are TinyStories (Eldan and Li, 2023): short children's stories written with a small vocabulary. The file is the public validation set, about 22,000 stories, under a CDLA-Sharing-1.0 license. The forest trains on 4,000 of them.

In [ ]:
STORY_URLS = [
    "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt",
    "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt?download=true",
]
EXPECTED_SHA256 = "94e431816c4cce81ff71e4408ff8d3bda9a42e8d2663986697c3954288cb38b4"


def download_stories():
    errors = []
    for url in STORY_URLS:
        request = urllib.request.Request(
            url,
            headers={"User-Agent": "ACC380K-next-word-demo/0.4 (classroom use)"},
        )
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                text = response.read().decode("utf-8")
        except Exception as error:
            errors.append(url + " — " + str(error))
            continue
        if "<|endoftext|>" not in text:
            errors.append(url + " — downloaded file does not look like TinyStories")
            continue
        return text
    raise RuntimeError(
        "Could not download TinyStories.\n" + "\n".join(errors)
        + "\nCheck that this Colab session has internet access, then run this cell again."
    )


story_text = download_stories()
if hashlib.sha256(story_text.encode("utf-8")).hexdigest() != EXPECTED_SHA256:
    print("This file differs from the one used to write this notebook. Results may differ.")


def tokenize(text):
    lowered = text.lower()
    lowered = lowered.replace("’", "'").replace("‘", "'")
    lowered = lowered.replace("“", " ").replace("”", " ").replace('"', " ")
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^a-z0-9\s]", lowered)


stories = [s.strip() for s in story_text.split("<|endoftext|>") if len(s.strip()) > 200]
train_token_lists = [tokenize(s) for s in stories[:4000]]
test_token_lists = [tokenize(s) for s in stories[8000:8400]]

print("Training stories:", len(train_token_lists))
print("Held-out stories:", len(test_token_lists))
print()
print("One story, as tokens:")
print(train_token_lists[1][:30])

## Each row is one guess

Slide a window along each story. The last four tokens are X. The token that comes next is y. The forest only learns the 800 most common tokens as labels. A rarer word in the context becomes `<UNK>`, and a row whose next word is rarer than that is left out.

Four words of memory cannot tell whether the story is about a girl or a boy, so the story would drift from *she* to *he*. One extra column fixes most of that: `recent_pronoun` is the most recent *he/him/his* or *she/her* in the last 60 tokens, or `none`.

Predict before you run: in the window `once upon a time`, what is y?

In [ ]:
VOCAB_SIZE = 800
token_counts = Counter()
for tokens in train_token_lists:
    token_counts.update(tokens)
labels = {token for token, count in token_counts.most_common(VOCAB_SIZE)}

SHE_WORDS = {"she", "her", "hers", "herself"}
HE_WORDS = {"he", "him", "his", "himself"}


def recent_pronoun(previous_tokens):
    for token in reversed(previous_tokens[-60:]):
        if token in SHE_WORDS:
            return "she"
        if token in HE_WORDS:
            return "he"
    return "none"


def features(previous_tokens):
    padded = [BOS] * K + list(previous_tokens)
    context = [t if t in labels or t == BOS else UNK for t in padded[-K:]]
    return context + [recent_pronoun(previous_tokens)]


FEATURE_COLUMNS = CONTEXT_COLUMNS + ["recent_pronoun"]


def build_table(token_lists, keep_unknown_targets):
    rows = []
    for tokens in token_lists:
        for index, next_word in enumerate(tokens):
            if next_word in labels or keep_unknown_targets:
                rows.append(features(tokens[:index]) + [next_word])
    return pd.DataFrame(rows, columns=FEATURE_COLUMNS + ["next_word"])


train_table = build_table(train_token_lists, keep_unknown_targets=False)
test_table = build_table(test_token_lists, keep_unknown_targets=True)

print("Training rows:", f"{len(train_table):,}")
train_table.iloc[[0, 1, 2, 3, 4, 40, 41, 42]]

## What the forest actually sees

A tree cannot split on the word `time`. It splits on numbers. One-hot encoding makes a 0/1 column for every word at every position, so `t_minus_1_time` is 1 when the previous token is `time` and 0 otherwise. Position is kept: `the dog` and `dog the` turn on different columns.

The cell below encodes the opening of one story and shows it as a DataFrame. The real table has over three thousand columns, almost all 0 in every row, so the display keeps only the columns that are 1 somewhere in these rows. Each row has exactly five 1s: one per context position, plus one for `recent_pronoun`.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore")
X_train = encoder.fit_transform(train_table[FEATURE_COLUMNS])
y_train = train_table["next_word"]
print("One-hot matrix:", f"{X_train.shape[0]:,}", "rows x", f"{X_train.shape[1]:,}", "columns")

# The opening of the second training story: "once upon a time , in a big forest"
start = sum(token in labels for token in train_token_lists[0])
shown = range(start, start + 6)
one_hot = pd.DataFrame(
    X_train[shown].toarray().astype(int),
    columns=encoder.get_feature_names_out(),
)
one_hot = one_hot.loc[:, one_hot.sum() > 0]
pd.set_option("display.max_columns", 40)
one_hot.insert(0, "next_word", y_train.iloc[shown].to_numpy())

print()
print("The same rows as words:")
print(train_table.iloc[shown].to_string(index=False))
print()
print("As the forest sees them (columns that are 1 somewhere in these rows):")
one_hot

## Fit the forest

Each tree may grow up to 10,000 leaves, enough room for phrases that recur, such as "once upon a time". `max_features=0.1` means each split looks at a random tenth of the columns, which keeps the fit fast and makes the trees differ.

This is the slow cell: about two to three minutes on a free Colab runtime.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=20,
    max_leaf_nodes=10000,
    min_samples_leaf=2,
    max_features=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
started = time.time()
forest.fit(X_train, y_train)
print("Fit seconds:", round(time.time() - started))

held_out_guess = forest.predict(encoder.transform(test_table[FEATURE_COLUMNS]))
accuracy = np.mean(held_out_guess == test_table["next_word"].to_numpy())
print(f"Top guess is the true next word on held-out stories: {accuracy:.1%}")

## The forest's vote for one context

Each tree drops the row down to a leaf and reports the share of each next word among the training rows in that leaf. `predict_proba` averages those shares across the 20 trees. The result is a ranked list of next words.

In [ ]:
forest.n_jobs = 1  # one row at a time is faster on a single core


def next_word_probabilities(previous_tokens):
    row = pd.DataFrame([features(previous_tokens)], columns=FEATURE_COLUMNS)
    return forest.predict_proba(encoder.transform(row))[0]


def vote_table(prompt):
    probabilities = next_word_probabilities(tokenize(prompt))
    ranked = pd.DataFrame({"next_word": forest.classes_, "probability": probabilities})
    ranked = ranked.sort_values("probability", ascending=False).head(5)
    return ranked.assign(percent=(ranked["probability"] * 100).round(1)).reset_index(drop=True)


print("After 'once upon a':")
print(vote_table("once upon a").to_string(index=False))
print()
print("After 'lily went to the':")
print(vote_table("lily went to the").to_string(index=False))

## Predict, append, shift, repeat

Start from a prompt. Then:

1. `predict_proba` scores every next word.
2. Take the top word.
3. Append it, slide the window, and repeat.

Always taking the top word gets stuck in a loop, such as *she was so happy and she was very happy. she was so happy and she was very happy…*. One rule prevents that: never write the same three-word phrase twice. If the top word would repeat a phrase, take the best word that does not. Large language models offer the same rule as a setting.

In [ ]:
# Try it: change PROMPT and rerun this cell. The forest stays the same.
PROMPT = "once upon a time"
NEW_TOKENS = 80


def write_story(prompt, n_new_tokens, temperature=0.0, top_k=5):
    # temperature 0 always takes the top word. Above 0, draw among the top_k words.
    rng = np.random.default_rng()
    words = tokenize(prompt)
    for _ in range(n_new_tokens):
        probabilities = next_word_probabilities(words)
        candidates = []
        for index in np.argsort(probabilities)[::-1]:
            candidate = str(forest.classes_[index])
            phrase = words[-2:] + [candidate]
            already_written = any(words[i:i + 3] == phrase for i in range(len(words) - 2))
            if not already_written:
                candidates.append(index)
            if len(candidates) == top_k:
                break
        choice = candidates[0]
        if temperature > 0:
            weights = probabilities[candidates] ** (1 / temperature)
            if weights.sum() > 0:
                choice = rng.choice(candidates, p=weights / weights.sum())
        words.append(str(forest.classes_[choice]))
    return words


def readable(tokens):
    text = " ".join("I" + t[1:] if t == "i" or t.startswith("i'") else t for t in tokens)
    text = re.sub(r" ([.,!?;:])", r"\1", text)
    return re.sub(r"(^|[.!?] )([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)


story = write_story(PROMPT, NEW_TOKENS)
print(readable(story))

## Read the story

Most sentences are grammatical, the character keeps her pronoun, and the story sounds like TinyStories. It still has no plot. Each word was picked from the last four tokens and one pronoun clue, so the forest cannot remember that Lily was already playing outside when it sends her to the park, and an odd phrase such as *she said, I want to play with her toys* can slip in.

The story is a continuation built from patterns. It is not a summary of any TinyStories example, and it is not something you would cite.

Run the cell again and you get the same story, word for word. Taking the top word every time is deterministic.

## Add randomness: temperature

Instead of always taking the top word, draw one of the five best words at random, using the forest's probabilities as weights. **Temperature** controls how much the draw favors the top word:

- Each weight is the probability raised to the power `1 / temperature`.
- `temperature = 1` uses the forest's probabilities as they are.
- Below 1 sharpens them: the top word wins more often. Near 0 it is the same as always taking the top word.
- Above 1 flattens them: lower-ranked words win more often, and the story gets more surprising and less sensible.

Every run draws new words, so rerun the cell and you get three new stories. ChatGPT exposes the same two settings to developers: a temperature and a cutoff for how many candidate tokens to consider.

In [ ]:
# Try it: change TEMPERATURE and rerun this cell. Each run gives new stories.
TEMPERATURE = 1

for story_number in range(1, 4):
    print("Temperature", TEMPERATURE, "| story", story_number)
    print(readable(write_story(PROMPT, NEW_TOKENS, temperature=TEMPERATURE)))
    print()

## What this shares with ChatGPT

At generation time, ChatGPT and this notebook do the same kind of step: read the tokens so far, turn them into a distribution over the next token, choose one, append it, repeat. The machinery that builds the distribution is different.

| | This notebook | A model such as ChatGPT |
|---|---|---|
| What one step predicts | The next word or punctuation mark | The next token, often a piece of a word |
| How a word is represented | A separate 0/1 column for each word at each position | A learned vector, an embedding, that places similar words near each other |
| How it uses earlier words | Only the ones we turned into columns: the last 4, plus one pronoun clue | Attention weighs every earlier token and learns which ones matter |
| How much context | 4 tokens | Thousands of tokens |
| How it is trained | Twenty decision trees on 4,000 short stories | A neural network on a very large collection of text |
| What training asks | Which next word follows this context? | The same question, at a much larger scale |

## Fluent is not the same as checked

A next-word objective rewards a plausible continuation. It does not audit the sentence. The story above can sound right and still invent an event.

Before you used a generated sentence about a company's revenue, you would go back to the records: the right entity, the right period, the calculation, and whether the words follow from that evidence. This forest never does that step. A larger language model does not do it either, unless a person or a separate check does.

## Try it

1. **Change the prompt.** In the story cell, set `PROMPT = "the little dog"` or `PROMPT = "tom and his mom"` and rerun that cell only.
2. **Turn up the temperature.** In the temperature cell, try `TEMPERATURE = 0.2`, then `1.0`, then `1.5`. At which setting do the stories stop making sense?
3. **Remove the loop guard.** In `write_story`, change `if not already_written:` to `if True:` and rerun the cell. How soon does the story start repeating?